<div class="alert alert-info" role="alert" style="padding:20px; margin-bottom:16px;">
  <div style="text-align:center;">
    <h1 style="margin:0;">Exploratory Data Analysis</h1>
    <div style="font-size:18px; margin-top:4px;">Amazing International Airlines Inc.</div>
    <div style="font-size:18px; margin-top:4px;">Clustering - Merge all perspectives</div>
    <hr style="margin:12px auto; width:220px;">
    <div style="font-size:14px; color:#6c757d;">Group 92 • Notebook • 2025/2026</div>
  </div>
</div>


This Project was done by:


Student Name    -   Mehmet Karaca;
student id      -   20250344;
contact email   -   20250344@novaims.unl.pt

Student Name    -   Duarte Gomes;
student id      -   20250017;
contact email   -   20250017@novaims.unl.pt

Student Name    -   Esra Salhi
student id      -   20250537
contact email   -   20250537@novaims.unl.pt

## Table of Contents



Add some contents here



# 1 Introduction <a id="introduction"></a>


## 1.1. Importing Libraries <a id="importing-libraries"></a>


In [53]:
# --- Standard Imports
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import matplotlib
import math
import matplotlib as mpl
from cycler import cycler
import geopandas as gpd
from matplotlib.lines import Line2D
from sklearn.impute import KNNImputer
from sklearn.cluster import DBSCAN, KMeans, AgglomerativeClustering
from sklearn.base import clone
from scipy.cluster.hierarchy import dendrogram, linkage



## 1.2. Loading and Reading Data <a id="loading-and-reading-data"></a>

In [55]:
# directory with raw CSV files
data_dir = Path("../data/cleanAndFeatureEngineered")

# list all CSV files
csv_files = list(data_dir.glob("*.csv"))
print(f"Found CSV files: {[f.name for f in csv_files]}")


Found CSV files: ['DM_AIAI_FlightsDB_Cleaned_Featured.csv', 'DM_AIAI_CustomerDB_Cleaned_Featured.csv']


In [56]:
# specify the files we want to load

customers_file = data_dir / "DM_AIAI_CustomerDB_Cleaned_Featured.csv"
flights_file   = data_dir / "DM_AIAI_FlightsDB_Cleaned_Featured.csv"

# load them with pandas
customers = pd.read_csv(customers_file)
flights   = pd.read_csv(flights_file)

print("Customers shape:", customers.shape)
print("Flights shape:", flights.shape)

Customers shape: (16375, 53)
Flights shape: (596664, 11)


## 1.3. Brief Preliminary Analysis <a id="brief-preliminary-analysis"></a>

In [57]:
customers.columns

Index(['Loyalty#', 'First Name', 'Last Name', 'Customer Name', 'Country',
       'City', 'Latitude', 'Longitude', 'Postal code', 'Income',
       'EnrollmentDateOpening', 'CancellationDate', 'Customer Lifetime Value',
       'IsActive', 'CustomerTenureDays', 'total_flights',
       'total_flights_with_companions', 'total_distance',
       'total_points_accumulated', 'total_points_redeemed',
       'total_cost_redeemed', 'average_distance_per_flight',
       'points_redemption_ratio', 'companion_flight_ratio',
       'LoyaltyStatus_Aurora', 'LoyaltyStatus_Nova', 'LoyaltyStatus_Star',
       'EnrollmentType_2021 Promotion', 'EnrollmentType_Standard',
       'Gender_female', 'Gender_male', 'Education_Bachelor',
       'Education_College', 'Education_Doctor',
       'Education_High School or Below', 'Education_Master',
       'Marital Status_Divorced', 'Marital Status_Married',
       'Marital Status_Single', 'Province or State_Alberta',
       'Province or State_British Columbia', 'Provin

In [58]:
flights.columns

Index(['Loyalty#', 'Year', 'Month', 'YearMonthDate', 'NumFlights',
       'NumFlightsWithCompanions', 'DistanceKM', 'PointsAccumulated',
       'PointsRedeemed', 'DollarCostPointsRedeemed', 'Season'],
      dtype='object')

In [7]:
customers.describe(include='number')

,Loyalty#,Latitude,Longitude,Income,Customer Lifetime Value,IsActive,CustomerTenureDays,total_flights,total_flights_with_companions,total_distance,total_points_accumulated,total_points_redeemed,total_cost_redeemed,average_distance_per_flight,points_redemption_ratio,companion_flight_ratio
count,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000,16375.000000
mean,549432.932641,47.174418,-91.838613,37721.547359,7982.088433,0.874931,1104.017282,160.262864,34.035603,289943.099212,28988.602947,8601.144116,84.999084,8053.974942,0.286192,0.219299
std,258720.547621,3.306921,22.245806,30342.177810,6841.119253,0.330807,741.228136,67.924837,22.216511,162257.383492,16223.028595,8707.423785,86.110464,4507.149549,0.410727,0.128028
min,100018.000000,42.984924,-135.056840,0.000000,1898.010000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,326276.000000,44.231171,-120.237660,0.000000,3979.390000,1.000000,384.000000,121.600000,16.000000,153607.450000,15356.795000,0.000000,0.000000,4266.875000,0.000000,0.150641
50%,550434.000000,46.087818,-79.383186,34115.000000,5781.020000,1.000000,1032.000000,173.000000,35.000000,333555.500000,33350.310000,6627.000000,65.000000,9265.430000,0.217670,0.221675
75%,771330.000000,49.282730,-74.596184,62375.000000,8936.970000,1.000000,1754.000000,213.800000,50.000000,414067.300000,41400.130000,13663.850000,134.950000,11501.870000,0.420562,0.288789
max,999986.000000,60.721188,-52.712578,99981.000000,83325.380000,1.000000,2466.000000,340.000000,119.000000,712729.600000,71264.460000,57527.800000,572.000000,19798.040000,16.125899,1.000000


# 2. Best Clustering for each perspective <a class="anchor" id="data-validity-checks"></a>

The best Clustering methods for each perspective was found in the previous notebooks. This methods will be applied again to put a label for the dataset customer. All entries will get 3 labels (One from each method)

In [ ]:
# Define feature categories from previous analysis

value_based_features = [
    'Customer Lifetime Value',
    'Income',
    'total_flights',
    'total_distance',
    'average_distance_per_flight',
    'total_points_accumulated',
    'total_cost_redeemed',
    'points_redemption_ratio',
    'IsActive',
    'EnrollmentType_Standard',
    'LoyaltyStatus_Nova',
    'LoyaltyStatus_Star'
]


demographic_features = [
    'Province or State_Newfoundland',
    'Province or State_Nova Scotia',
    'Province or State_Ontario',
    'Province or State_Prince Edward Island',
    'Province or State_Quebec',
    'Province or State_Saskatchewan',
    'Province or State_Yukon',
    'Location Code_Suburban',
    'Location Code_Urban',
    'Latitude',
    'Longitude',
    'Gender_male',
    'Education_College',
    'Education_High School or Below',
    'Education_Master',
    'Education_Doctor',
    'Income',
    'Marital Status_Married',
    'Marital Status_Single',
]


behavioral_features = [
    'total_flights',
    'total_flights_with_companions',
    'companion_flight_ratio',
    'average_distance_per_flight',
    'total_distance',
    'points_redemption_ratio',
    'IsActive',
    'CustomerTenureDays',
    'LoyaltyStatus_Nova',
    'LoyaltyStatus_Star'
]


We need the best Clustering approach for each Perspective.

- Value Based Perspective will use Mean Shift Clustering.
- Demographics Perspective will use Hierarchical Clustering.
- Behavioral Perspective will use Mean Shift Clustering

## 2.1 Value Based 